# Stage 3B — Panel Remediation

**Purpose:** correct three independent data-quality defects discovered in
`raw_price_panel_consolidated_v1_2026-07-15.parquet`, producing a new,
versioned `v2` panel. v1 is **never modified**.

---

## Why this notebook exists

Stage 3 built and validated v1. It passed all 10 checks in
`stage3_final_confidence_check.py`. Those checks were correct — they were
also **threshold-based**, and this contamination is **sub-threshold by
construction**.

The ±30% review list caught **193 of ~21,472 artifact returns (0.9%)**.
It is not a safety net; it is a *magnitude* filter. A phantom price sitting
near the resuming real price produces a *small* artifact **because it is a
copy of a nearby price**. That check could not have worked.

What found it was **structural** — searching for a known fabrication
signature rather than for large numbers.

> **Lesson for the manifest: thresholds catch loud errors; only structure
> catches quiet ones.**

---

## The three mechanisms

| # | Mechanism | Rows | Detection | Action |
|---|---|---|---|---|
| 1 | **Phantom rows** — yfinance invents a row (prior close copied into OHLC, `Volume==0`) when a stock does not trade | 57,876 | `O==H==L==C AND Volume==0` (exact, not heuristic) | NaN the OHLC, keep `Volume=0`, keep the row |
| 2 | **Unadjusted split splices** — split applied to post-history but not pre-history | 8 | Observed ratio ≈ declared split ratio | NaN the boundary row's OHLC |
| 3 | **Isolated bad prints** — one row at N× scale, reverses next day | 6 | Hand-verified, hardcoded | NaN the OHLC |

**Total: 68,587 rows (2.06% of panel). 100% yfinance. Zero Bloomberg.**

---

## The governing principle

> **When we do not know the price, say so. Never substitute a plausible number.**

This is not fastidiousness — it is the reason the problem exists. yfinance's
phantom rows *are* a plausible substitute for missing data. They look like
prices, pass every schema check, and have been silently wrong since Stage 3.
The fix is not a better guess. It is `NaN`.

No interpolation. No forward-fill. No winsorizing. No deletion.

---

## Decisions ratified before writing this notebook

- **Fix at the raw panel, not the feature layer.** The rows are wrong at the
  source. Masking downstream would leave fabricated prices in the raw file for
  every future consumer to remember to handle. That is a landmine.
- **v1 is superseded, not deleted.** Overwriting it would destroy the evidence
  that found this and make "which panel produced that result?" unanswerable.
- **Keep phantom rows, mask their prices.** Deleting rows would silently change
  what "the prior trading row" means and corrupt `days_since_prev_trade`.
- **Keep `Volume = 0` on phantom rows.** Zero shares traded is real
  information. Q7 established yfinance encodes no-trade as `0` while Bloomberg
  uses `NaN` — do not destroy that distinction.
- **Splices: NaN the boundary, do NOT rescale.** Memory fact #1 — a prior
  custom split-adjustment attempt double-adjusted prices and was discarded, its
  output deleted from Drive. A rescale-only-these-8 layer is exactly the thing
  that gets partially re-applied later, silently. 8 nulled returns out of 3.3M
  is the cheaper error.
- **Bad prints: 6 hardcoded rows, no detector.** The shape rule worked here,
  but on a genuinely volatile stock a real move plus a real reversal would trip
  it. Six auditable rows beat a heuristic that silently nulls data.
- **No re-pull. No Bloomberg migration.** 8 bad splits in 582 is not vendor
  failure. Settled.

---

## Notebook order

| Cell | What | Gate |
|---|---|---|
| 0 | Mount + verify inputs | — |
| 1 | **Impact preview (READ-ONLY)** | **You approve the ticker list before Cell 2 runs** |
| 2 | Build v2 | Counts must match the diagnostic exactly: 57,876 / 8 / 6 |
| 3 | v2 diagnostic | Zero phantoms remain; Bloomberg bit-identical |

**Do not run Cell 2 until you have read Cell 1's output and approved it.**

## Cell 0 — Mount Drive & verify inputs

Read-only. Run once per Colab session.

In [1]:
# =============================================================================
# STAGE 3B / CELL 0 — Mount Drive + verify inputs
# READ-ONLY. Writes nothing.
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', 60)

BASE = Path('/content/drive/MyDrive/moonshot_compounding')

print(f"\nProject root exists: {BASE.exists()}  ->  {BASE}")
if not BASE.exists():
    raise FileNotFoundError(
        f"{BASE} not found. Check the path before running anything else — "
        "do not proceed on a guessed path."
    )

# --- The single input this notebook reads -----------------------------------
V1_PANEL = BASE / 'raw' / 'prices' / 'raw_price_panel_consolidated_v1_2026-07-15.parquet'
CORP_DIR = BASE / 'corporate_actions' / 'batches'

print()
if V1_PANEL.exists():
    print(f"  OK      v1 panel      : {V1_PANEL.stat().st_size/1e6:,.1f} MB")
else:
    raise FileNotFoundError(f"v1 panel not found at {V1_PANEL}. STOP.")

if CORP_DIR.exists():
    n = len(list(CORP_DIR.glob('*.parquet')))
    print(f"  OK      corp actions  : {n:,} parquet files")
else:
    raise FileNotFoundError(f"{CORP_DIR} not found. STOP.")

# --- Output dirs -------------------------------------------------------------
for d in [BASE / 'raw' / 'prices', BASE / 'validation']:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  READY   {d.relative_to(BASE)}")

# --- Refuse-to-overwrite pre-check ------------------------------------------
# Convention: derived files are timestamped and NEVER overwritten.
existing_v2 = sorted((BASE / 'raw' / 'prices').glob('raw_price_panel_consolidated_v2_*.parquet'))
print()
if existing_v2:
    print(f"  NOTE: {len(existing_v2)} v2 panel(s) already exist:")
    for p in existing_v2:
        print(f"        {p.name}")
    print("        Cell 2 will REFUSE to overwrite. Rename or remove deliberately")
    print("        if you intend to rebuild.")
else:
    print("  No prior v2 panel present — clean slate.")

print("\nMount complete. Cell 1 (impact preview) is safe to run.")

Mounted at /content/drive

Project root exists: True  ->  /content/drive/MyDrive/moonshot_compounding

  OK      v1 panel      : 90.4 MB
  OK      corp actions  : 475 parquet files
  READY   raw/prices
  READY   validation

  No prior v2 panel present — clean slate.

Mount complete. Cell 1 (impact preview) is safe to run.


## Cell 1 — Impact preview (READ-ONLY) — **THIS IS THE GATE**

Nothing is written. This cell answers one question:

> **Which tickers will masking push below the 120-day floor for `realized_vol_120`?**

Masking phantoms means some tickers lose usable history. **This is correct** —
a stock that did not trade has no volatility, and reporting one would be
fiction. It is the same rule that retired `DENABANK.NS` in Stage 3 (6 valid
closes out of 226).

But it must be **predicted before it is measured**, not rationalised after.

**Read the output. Approve the list. Only then run Cell 2.**

In [2]:
# =============================================================================
# STAGE 3B / CELL 1 — IMPACT PREVIEW  (READ-ONLY — THE GATE)
# Writes nothing. Mutates nothing.
#
# THE QUESTION
#   Masking will push some tickers below the 120-day minimum needed for the
#   longest planned feature (realized_vol_120). Which ones, and how many?
#
# WHY IT IS A GATE
#   Memory fact #11: NaN percentage alone is misleading. 97% NaN on a 226-row
#   file leaves ~6 usable days (unusable); 5% NaN on a 4,000-row file leaves
#   ~3,800 (fine). ALWAYS check the ABSOLUTE valid-row count against the
#   longest planned rolling window (120), never the percentage.
#   This is exactly how DENABANK.NS was correctly retired in Stage 3.
#
# NOTHING IS DECIDED HERE. The numbers are shown; you approve; Cell 2 acts.
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/content/drive/MyDrive/moonshot_compounding')
V1_PANEL = BASE / 'raw' / 'prices' / 'raw_price_panel_consolidated_v1_2026-07-15.parquet'
CORP_DIR = BASE / 'corporate_actions' / 'batches'

MIN_VALID_ROWS = 120   # longest planned rolling window (realized_vol_120)

print("=" * 110)
print("STAGE 3B / CELL 1 — IMPACT PREVIEW  (read-only)")
print("=" * 110)

px = pd.read_parquet(V1_PANEL)
px['Date'] = pd.to_datetime(px['Date'])
px = px.sort_values(['ticker', 'Date']).reset_index(drop=True)

print(f"\n  v1 rows    : {len(px):,}")
print(f"  v1 tickers : {px['ticker'].nunique():,}   (expected 1,069)")
assert px['ticker'].nunique() == 1069, "Expected 1,069 tickers. STOP."

# =============================================================================
# Identify all three mechanisms — same logic Cell 2 will use.
# Defined ONCE here so the preview and the build cannot drift apart.
# =============================================================================

# --- MECHANISM 1: phantom rows ----------------------------------------------
# EXACT signature, not a heuristic:
#   Open == High == Low == Close  AND  Volume == 0
# Bloomberg has ZERO such rows across 255,727 — this is a yfinance artifact,
# not a market phenomenon.
#
# NOTE the Volume==0 conjunct is doing real work:
#   - flat OHLC with Volume > 0  = a stock that genuinely opened, traded and
#     closed at one price (25,493 rows). NOT touched.
#   - zero volume with a price RANGE = odd, but not the fabrication signature
#     (363 rows). NOT touched.
px['_phantom'] = (
    (px['Open'] == px['High']) &
    (px['High'] == px['Low']) &
    (px['Low'] == px['Close']) &
    px['Open'].notna() &
    (px['Volume'] == 0)
)

# --- MECHANISM 2: unadjusted split splices (8 named events) -----------------
# Identified in the contamination diagnostic by comparing the OBSERVED price
# ratio across the boundary against the DECLARED split ratio, with NO tolerance
# band (Q4's 20% band was wider than the gap between hypotheses for small
# ratios — that is precisely why 6 of these 8 were missed).
SPLICE_EVENTS = [
    ('GREENPLY.NS',   '2016-01-06', 5.000000),
    ('MVL.NS',        '2010-11-16', 2.000000),
    ('TTML.NS',       '2013-08-07', 1.133333),
    ('IGARASHI.NS',   '2018-09-27', 1.123762),
    ('KTKBANK.NS',    '2020-03-17', 1.100000),
    ('KARURVYSYA.NS', '2018-08-14', 1.100000),
    ('JISLJALEQS.NS', '2011-11-08', 1.050000),
    ('AJRINFRA.NS',   '2013-01-23', 1.029412),
]
splice_keys = {(t, pd.Timestamp(d)) for t, d, _ in SPLICE_EVENTS}
px['_splice'] = [ (t, d) in splice_keys for t, d in zip(px['ticker'], px['Date']) ]

# --- MECHANISM 3: isolated bad prints (6 named rows) ------------------------
# Hand-verified. Hardcoded deliberately — NO detector in the pipeline. The
# shape rule (>50% then >33% reversal) worked here, but on a genuinely volatile
# stock a real move plus a real reversal would trip it. Six auditable rows beat
# a heuristic that silently nulls data.
BAD_PRINTS = [
    ('THYROCARE.NS', '2019-10-27'),
    ('THYROCARE.NS', '2020-11-14'),
    ('INDIAMART.NS', '2019-10-27'),
    ('INDIAMART.NS', '2020-11-14'),
    ('MVL.NS',       '2019-10-10'),
    ('MVL.NS',       '2019-12-20'),
]
print_keys = {(t, pd.Timestamp(d)) for t, d in BAD_PRINTS}
px['_badprint'] = [ (t, d) in print_keys for t, d in zip(px['ticker'], px['Date']) ]

px['_to_mask'] = px['_phantom'] | px['_splice'] | px['_badprint']

# =============================================================================
# [1] Reconcile against the contamination diagnostic — expected counts
# =============================================================================
print("\n[1] RECONCILIATION AGAINST THE CONTAMINATION DIAGNOSTIC")
print("    (These are stated in advance. A mismatch means the panel on disk is")
print("     not the one the diagnostic ran against — STOP if so.)")

exp = {'phantom': 57876, 'splice': 8, 'badprint': 6}
act = {
    'phantom': int(px['_phantom'].sum()),
    'splice': int(px['_splice'].sum()),
    'badprint': int(px['_badprint'].sum()),
}
ok = True
for k in exp:
    match = act[k] == exp[k]
    ok &= match
    print(f"    {k:<10}: expected {exp[k]:>7,}   found {act[k]:>7,}   "
          f"{'OK' if match else '*** MISMATCH ***'}")

print(f"\n    total rows to mask : {int(px['_to_mask'].sum()):,}")
print(f"    (phantom + splice + badprint, deduplicated — a row could in")
print(f"     principle be flagged by more than one mechanism)")

if not ok:
    print("\n    *** COUNTS DO NOT MATCH. Do NOT proceed to Cell 2 until this")
    print("    *** is understood. Either the panel differs from the one the")
    print("    *** diagnostic ran against, or the detection logic has drifted.")

# =============================================================================
# [2] Per-ticker usability impact — THE NUMBER THAT MATTERS
# =============================================================================
print("\n[2] PER-TICKER USABILITY IMPACT")
print(f"    Threshold: a ticker needs >= {MIN_VALID_ROWS} valid Close values to")
print(f"    support realized_vol_120 (the longest planned feature).")

# BEFORE: valid = Close is non-null (v1's own definition of usable)
# AFTER : valid = Close is non-null AND the row is not being masked
px['_valid_before'] = px['Close'].notna()
px['_valid_after'] = px['Close'].notna() & ~px['_to_mask']

impact = px.groupby('ticker').agg(
    rows=('Date', 'size'),
    valid_before=('_valid_before', 'sum'),
    valid_after=('_valid_after', 'sum'),
    phantom=('_phantom', 'sum'),
    source=('source', 'first'),
    first_date=('Date', 'min'),
    last_date=('Date', 'max'),
)
impact['lost'] = impact['valid_before'] - impact['valid_after']
impact['pct_lost'] = 100 * impact['lost'] / impact['valid_before'].replace(0, np.nan)
impact['usable_before'] = impact['valid_before'] >= MIN_VALID_ROWS
impact['usable_after'] = impact['valid_after'] >= MIN_VALID_ROWS
impact['CROSSES_THRESHOLD'] = impact['usable_before'] & ~impact['usable_after']

print(f"\n    tickers usable BEFORE masking : {impact['usable_before'].sum():,} / {len(impact):,}")
print(f"    tickers usable AFTER  masking : {impact['usable_after'].sum():,} / {len(impact):,}")
print(f"    tickers CROSSING the threshold: {impact['CROSSES_THRESHOLD'].sum():,}")

# =============================================================================
# [3] THE APPROVAL LIST
# =============================================================================
print("\n[3] *** THE APPROVAL LIST — tickers that lose usability ***")
crossing = impact[impact['CROSSES_THRESHOLD']]
if len(crossing) == 0:
    print("\n    NONE. No ticker falls below the 120-day floor as a result of")
    print("    masking. The usable universe is unchanged.")
else:
    print(f"\n    {len(crossing):,} ticker(s) will fall below {MIN_VALID_ROWS} valid rows.")
    print("    These become unusable for realized_vol_120 — which is CORRECT")
    print("    (a stock that did not trade has no volatility), but it is a real")
    print("    reduction in the usable universe. Approve it explicitly.\n")
    print(crossing[['rows', 'valid_before', 'valid_after', 'lost', 'pct_lost',
                    'source', 'first_date', 'last_date']]
          .sort_values('valid_after')
          .to_string(float_format=lambda x: f"{x:,.1f}"))

# =============================================================================
# [4] Near-threshold watch list — not lost, but close
# =============================================================================
print(f"\n[4] NEAR-THRESHOLD WATCH LIST (survive, but with < 2x the floor)")
near = impact[impact['usable_after'] & (impact['valid_after'] < 2 * MIN_VALID_ROWS)]
print(f"    tickers with {MIN_VALID_ROWS}-{2*MIN_VALID_ROWS} valid rows after masking: {len(near):,}")
if len(near):
    print("    (These survive but will have very few realized_vol_120 observations.")
    print("     Not a decision for this cell — flagged so it is not a surprise later.)\n")
    print(near[['rows', 'valid_before', 'valid_after', 'lost', 'pct_lost', 'source']]
          .sort_values('valid_after')
          .head(30).to_string(float_format=lambda x: f"{x:,.1f}"))

# =============================================================================
# [5] Worst-hit tickers by volume of masking
# =============================================================================
print("\n[5] WORST-HIT TICKERS BY ROWS MASKED (top 30)")
print("    Predicted before running: 25 tickers >25% phantom, 3 tickers >50%.")
print(impact[impact['lost'] > 0].nlargest(30, 'pct_lost')
      [['rows', 'valid_before', 'valid_after', 'lost', 'pct_lost',
        'usable_after', 'source']]
      .to_string(float_format=lambda x: f"{x:,.1f}"))

print(f"\n    tickers losing >0 rows   : {(impact['lost'] > 0).sum():,}")
for t in [1, 5, 10, 25, 50]:
    print(f"    tickers losing >{t:2d}% rows : {(impact['pct_lost'] > t).sum():,}")

# =============================================================================
# [6] Bloomberg must be untouched
# =============================================================================
print("\n[6] BLOOMBERG INTEGRITY CHECK")
bbg_masked = px[px['_to_mask'] & (px['source'] == 'bloomberg')]
print(f"    Bloomberg rows in panel     : {(px['source'] == 'bloomberg').sum():,}")
print(f"    Bloomberg rows to be masked : {len(bbg_masked):,}   (MUST be 0)")
if len(bbg_masked) > 0:
    print("\n    *** Bloomberg rows are being masked. Every mechanism found was")
    print("    *** 100% yfinance. This is unexpected — investigate before Cell 2.")
    print(bbg_masked[['Date','ticker','Open','High','Low','Close','Volume',
                      '_phantom','_splice','_badprint']].head(20).to_string(index=False))

# =============================================================================
# [7] Coverage impact by year
# =============================================================================
print("\n[7] COVERAGE IMPACT BY YEAR")
px['_yr'] = px['Date'].dt.year
yr = px.groupby('_yr').agg(
    rows=('Date', 'size'),
    valid_before=('_valid_before', 'sum'),
    valid_after=('_valid_after', 'sum'),
    masked=('_to_mask', 'sum'),
)
yr['pct_valid_before'] = 100 * yr['valid_before'] / yr['rows']
yr['pct_valid_after'] = 100 * yr['valid_after'] / yr['rows']
yr['pct_masked'] = 100 * yr['masked'] / yr['rows']
print(yr.to_string(float_format=lambda x: f"{x:,.2f}"))

print("\n" + "=" * 110)
print("IMPACT PREVIEW COMPLETE — nothing written.")
print()
print("READ SECTION [3]. That is the approval list.")
print("Do NOT run Cell 2 until you have approved it.")
print("=" * 110)

STAGE 3B / CELL 1 — IMPACT PREVIEW  (read-only)

  v1 rows    : 3,332,879
  v1 tickers : 1,069   (expected 1,069)

[1] RECONCILIATION AGAINST THE CONTAMINATION DIAGNOSTIC
    (These are stated in advance. A mismatch means the panel on disk is
     not the one the diagnostic ran against — STOP if so.)
    phantom   : expected  57,876   found  57,876   OK
    splice    : expected       8   found       8   OK
    badprint  : expected       6   found       6   OK

    total rows to mask : 57,890
    (phantom + splice + badprint, deduplicated — a row could in
     principle be flagged by more than one mechanism)

[2] PER-TICKER USABILITY IMPACT
    Threshold: a ticker needs >= 120 valid Close values to
    support realized_vol_120 (the longest planned feature).

    tickers usable BEFORE masking : 1,069 / 1,069
    tickers usable AFTER  masking : 1,069 / 1,069
    tickers CROSSING the threshold: 0

[3] *** THE APPROVAL LIST — tickers that lose usability ***

    NONE. No ticker falls below 

## Cell 2 — Build v2  ⚠️ **WRITES**

**Do not run until Cell 1's approval list is approved.**

This is the only cell in the notebook that writes. It:

1. Reads v1 (never modifies it)
2. Applies the three fixes — NaN the OHLC, keep the row, keep `Volume`
3. Adds `price_status` provenance on every row
4. Writes `raw_price_panel_consolidated_v2_<date>.parquet`
5. Writes a full change log to `validation/`

**It refuses to run if a v2 file already exists.** Timestamped, never
overwritten — the non-negotiable convention.

In [3]:
# =============================================================================
# STAGE 3B / CELL 2 — BUILD v2 PANEL   *** THIS CELL WRITES ***
#
# READS  : raw_price_panel_consolidated_v1_2026-07-15.parquet   (NEVER modified)
# WRITES : raw_price_panel_consolidated_v2_<date>.parquet       (new file)
#          validation/remediation_change_log_<date>.csv
#          validation/remediation_summary_<date>.csv
#
# WHAT IT DOES
#   Mechanism 1 — phantom rows (57,876): NaN the OHLC. KEEP Volume=0. KEEP row.
#   Mechanism 2 — split splices (8)    : NaN the OHLC on the boundary row.
#   Mechanism 3 — bad prints (6)       : NaN the OHLC.
#
# WHAT IT DOES NOT DO
#   - Does not delete rows. Deleting would silently change what "the prior
#     trading row" means and corrupt days_since_prev_trade.
#   - Does not interpolate, forward-fill, or winsorize. Every one of those
#     replaces "unknown" with a plausible lie — the exact disease.
#   - Does not rescale split pre-history. Memory fact #1: a prior custom
#     split-adjustment attempt double-adjusted prices and was discarded, its
#     output deleted from Drive. Do NOT resurrect that logic.
#   - Does not touch v1.
#
# PROVENANCE
#   Every row carries price_status:
#     'observed'             — untouched
#     'phantom_masked'       — yfinance fabricated row
#     'split_splice_masked'  — unadjusted split boundary
#     'bad_print_masked'     — isolated bad print
#   No silent handling. Every masked row states its reason.
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

BASE = Path('/content/drive/MyDrive/moonshot_compounding')
V1_PANEL = BASE / 'raw' / 'prices' / 'raw_price_panel_consolidated_v1_2026-07-15.parquet'
VALIDATION_DIR = BASE / 'validation'
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

BUILD_DATE = datetime.now().strftime('%Y-%m-%d')
V2_PANEL = BASE / 'raw' / 'prices' / f'raw_price_panel_consolidated_v2_{BUILD_DATE}.parquet'
CHANGE_LOG = VALIDATION_DIR / f'remediation_change_log_{BUILD_DATE}.csv'
SUMMARY = VALIDATION_DIR / f'remediation_summary_{BUILD_DATE}.csv'

print("=" * 110)
print("STAGE 3B / CELL 2 — BUILD v2   *** WRITES ***")
print("=" * 110)

# --- Refuse to overwrite -----------------------------------------------------
# Non-negotiable: raw/derived files are timestamped and NEVER overwritten.
for p in [V2_PANEL, CHANGE_LOG, SUMMARY]:
    if p.exists():
        raise FileExistsError(
            f"{p} already exists. This cell does not overwrite. "
            "If you intend to rebuild, rename or remove the existing file "
            "deliberately — do not let a re-run silently replace it."
        )

# --- 1. Load v1 --------------------------------------------------------------
print("\n[1] LOADING v1  (read-only — v1 is never modified)")
px = pd.read_parquet(V1_PANEL)
px['Date'] = pd.to_datetime(px['Date'])
px = px.sort_values(['ticker', 'Date']).reset_index(drop=True)

V1_ROWS = len(px)
V1_TICKERS = px['ticker'].nunique()
print(f"  rows    : {V1_ROWS:,}")
print(f"  tickers : {V1_TICKERS:,}")
print(f"  columns : {list(px.columns)}")
assert V1_ROWS == 3_332_879, f"v1 row count is {V1_ROWS:,}, expected 3,332,879. STOP."
assert V1_TICKERS == 1069, f"v1 ticker count is {V1_TICKERS:,}, expected 1,069. STOP."

# --- 2. Identify — IDENTICAL logic to Cell 1 --------------------------------
print("\n[2] IDENTIFYING CONTAMINATION")

px['_phantom'] = (
    (px['Open'] == px['High']) &
    (px['High'] == px['Low']) &
    (px['Low'] == px['Close']) &
    px['Open'].notna() &
    (px['Volume'] == 0)
)

SPLICE_EVENTS = [
    ('GREENPLY.NS',   '2016-01-06', 5.000000),
    ('MVL.NS',        '2010-11-16', 2.000000),
    ('TTML.NS',       '2013-08-07', 1.133333),
    ('IGARASHI.NS',   '2018-09-27', 1.123762),
    ('KTKBANK.NS',    '2020-03-17', 1.100000),
    ('KARURVYSYA.NS', '2018-08-14', 1.100000),
    ('JISLJALEQS.NS', '2011-11-08', 1.050000),
    ('AJRINFRA.NS',   '2013-01-23', 1.029412),
]
splice_keys = {(t, pd.Timestamp(d)) for t, d, _ in SPLICE_EVENTS}
px['_splice'] = [ (t, d) in splice_keys for t, d in zip(px['ticker'], px['Date']) ]

BAD_PRINTS = [
    ('THYROCARE.NS', '2019-10-27'),
    ('THYROCARE.NS', '2020-11-14'),
    ('INDIAMART.NS', '2019-10-27'),
    ('INDIAMART.NS', '2020-11-14'),
    ('MVL.NS',       '2019-10-10'),
    ('MVL.NS',       '2019-12-20'),
]
print_keys = {(t, pd.Timestamp(d)) for t, d in BAD_PRINTS}
px['_badprint'] = [ (t, d) in print_keys for t, d in zip(px['ticker'], px['Date']) ]

n_phantom = int(px['_phantom'].sum())
n_splice = int(px['_splice'].sum())
n_badprint = int(px['_badprint'].sum())

print(f"  phantom rows    : {n_phantom:,}   (expected 57,876)")
print(f"  splice rows     : {n_splice:,}   (expected 8)")
print(f"  bad print rows  : {n_badprint:,}   (expected 6)")

# HARD GATE. If the panel on disk is not the one the diagnostic ran against,
# masking it blind would be worse than not masking at all.
assert n_phantom == 57876, (
    f"Phantom count is {n_phantom:,}, expected 57,876. The panel on disk is not "
    "the one the contamination diagnostic ran against, or the detection logic "
    "has drifted. STOP — do not mask blind."
)
assert n_splice == 8, f"Splice count is {n_splice}, expected 8. STOP."
assert n_badprint == 6, f"Bad print count is {n_badprint}, expected 6. STOP."

# Bloomberg must be untouched. Every mechanism found was 100% yfinance.
bbg_hit = px[(px['_phantom'] | px['_splice'] | px['_badprint']) &
             (px['source'] == 'bloomberg')]
print(f"  Bloomberg rows affected : {len(bbg_hit):,}   (MUST be 0)")
assert len(bbg_hit) == 0, (
    f"{len(bbg_hit):,} Bloomberg rows flagged for masking. Every mechanism found "
    "was 100% yfinance. This is unexpected — STOP and investigate."
)

# --- 3. Build the change log BEFORE mutating --------------------------------
# Capture original values so every change is reversible and auditable.
print("\n[3] CAPTURING CHANGE LOG (before values)")

px['_to_mask'] = px['_phantom'] | px['_splice'] | px['_badprint']

# A row could in principle be flagged by more than one mechanism. Assign a
# single reason with an explicit precedence rather than letting it be ambiguous.
px['price_status'] = 'observed'
px.loc[px['_phantom'], 'price_status'] = 'phantom_masked'
px.loc[px['_splice'], 'price_status'] = 'split_splice_masked'
px.loc[px['_badprint'], 'price_status'] = 'bad_print_masked'

overlap = px[(px['_phantom'].astype(int) + px['_splice'].astype(int) +
              px['_badprint'].astype(int)) > 1]
print(f"  rows flagged by >1 mechanism : {len(overlap):,}")
if len(overlap):
    print("    (precedence applied: phantom < splice < bad_print. Listed here so")
    print("     the double-classification is visible, not silently resolved.)")
    print(overlap[['Date','ticker','_phantom','_splice','_badprint','price_status']]
          .to_string(index=False))

change_log = px.loc[px['_to_mask'], [
    'Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
    'source', 'price_status'
]].copy()
change_log = change_log.rename(columns={
    'Open': 'orig_Open', 'High': 'orig_High',
    'Low': 'orig_Low', 'Close': 'orig_Close',
})
change_log['action'] = 'OHLC_set_to_NaN'
change_log['volume_retained'] = True
print(f"  change log rows : {len(change_log):,}")

# --- 4. Apply the mask -------------------------------------------------------
print("\n[4] APPLYING THE MASK")
print("  Setting Open/High/Low/Close to NaN on masked rows.")
print("  Volume is RETAINED — zero shares traded is real information, and Q7")
print("  established yfinance encodes no-trade as 0 while Bloomberg uses NaN.")
print("  Rows are RETAINED — deleting them would corrupt 'prior trading row'.")

before_close_valid = int(px['Close'].notna().sum())

for col in ['Open', 'High', 'Low', 'Close']:
    px.loc[px['_to_mask'], col] = np.nan

after_close_valid = int(px['Close'].notna().sum())

print(f"\n  valid Close BEFORE : {before_close_valid:,}")
print(f"  valid Close AFTER  : {after_close_valid:,}")
print(f"  delta              : {before_close_valid - after_close_valid:,}   "
      f"(should equal rows masked that had a valid Close)")

# --- 5. Verify the mask actually took ---------------------------------------
print("\n[5] VERIFYING THE MASK")
residual = (
    (px['Open'] == px['High']) & (px['High'] == px['Low']) &
    (px['Low'] == px['Close']) & px['Open'].notna() & (px['Volume'] == 0)
)
print(f"  residual phantom rows : {int(residual.sum()):,}   (MUST be 0)")
assert residual.sum() == 0, "Phantom rows survived the mask. STOP."

masked_with_price = px[px['_to_mask'] & px[['Open','High','Low','Close']].notna().any(axis=1)]
print(f"  masked rows still holding a price : {len(masked_with_price):,}   (MUST be 0)")
assert len(masked_with_price) == 0, "A masked row retained a price. STOP."

assert len(px) == V1_ROWS, f"Row count changed: {len(px):,} vs {V1_ROWS:,}. STOP."
assert px['ticker'].nunique() == V1_TICKERS, "Ticker count changed. STOP."
print(f"  row count preserved    : {len(px):,}  OK")
print(f"  ticker count preserved : {px['ticker'].nunique():,}  OK")

# --- 6. Emit -----------------------------------------------------------------
print("\n[6] EMITTING v2")

out_cols = [
    'Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
    'adj_close_dividend_yfinance_unused', 'source', 'price_status',
]
out_cols = [c for c in out_cols if c in px.columns]
v2 = px[out_cols].copy()

print(f"  schema : {list(v2.columns)}")
print(f"\n  price_status distribution:")
print(v2['price_status'].value_counts().to_string())

v2.to_parquet(V2_PANEL, index=False)
print(f"\n  -> written: {V2_PANEL.name}")
print(f"     {len(v2):,} rows, {v2['ticker'].nunique():,} tickers")

change_log.sort_values(['ticker', 'Date']).to_csv(CHANGE_LOG, index=False)
print(f"  -> written: {CHANGE_LOG.name}  ({len(change_log):,} rows)")

summary = pd.DataFrame([
    dict(mechanism='phantom_rows', rows=n_phantom,
         detection='Open==High==Low==Close AND Volume==0',
         action='OHLC set to NaN; Volume and row retained',
         note='yfinance fabricates a row when a stock does not trade. '
              'Bloomberg has zero such rows across 255,727.'),
    dict(mechanism='unadjusted_split_splices', rows=n_splice,
         detection='observed close ratio ~= declared split ratio at ex-date',
         action='OHLC set to NaN on the boundary row',
         note='Pre-split history remains on a different scale. Returns WITHIN '
              'the pre-split window are correct (ratios are scale-invariant); '
              'only the boundary is broken and it is nulled. Price LEVELS '
              'across the split are not comparable for these 8 tickers.'),
    dict(mechanism='isolated_bad_prints', rows=n_badprint,
         detection='hand-verified; hardcoded (no detector in pipeline)',
         action='OHLC set to NaN',
         note='Shape rule worked here but would trip on genuinely volatile '
              'stocks. Six auditable rows beat a silent heuristic.'),
])
summary['source_file'] = V1_PANEL.name
summary['output_file'] = V2_PANEL.name
summary['build_date'] = BUILD_DATE
summary.to_csv(SUMMARY, index=False)
print(f"  -> written: {SUMMARY.name}")

print("\n" + "=" * 110)
print("v2 BUILD COMPLETE — run Cell 3 (diagnostic) before using this file.")
print("v1 remains on disk, unmodified, superseded but not deleted.")
print("A build that ran without error is NOT a build that is correct.")
print("=" * 110)

STAGE 3B / CELL 2 — BUILD v2   *** WRITES ***

[1] LOADING v1  (read-only — v1 is never modified)
  rows    : 3,332,879
  tickers : 1,069
  columns : ['Date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'adj_close_dividend_yfinance_unused', 'source']

[2] IDENTIFYING CONTAMINATION
  phantom rows    : 57,876   (expected 57,876)
  splice rows     : 8   (expected 8)
  bad print rows  : 6   (expected 6)
  Bloomberg rows affected : 0   (MUST be 0)

[3] CAPTURING CHANGE LOG (before values)
  rows flagged by >1 mechanism : 0
  change log rows : 57,890

[4] APPLYING THE MASK
  Setting Open/High/Low/Close to NaN on masked rows.
  Volume is RETAINED — zero shares traded is real information, and Q7
  established yfinance encodes no-trade as 0 while Bloomberg uses NaN.
  Rows are RETAINED — deleting them would corrupt 'prior trading row'.

  valid Close BEFORE : 3,317,223
  valid Close AFTER  : 3,259,333
  delta              : 57,890   (should equal rows masked that had a valid Close)

[5]

## Cell 3 — v2 Diagnostic (gate)

Read-only. Writes nothing.

**Nothing downstream consumes v2 until this passes.**

Checks:

- **A** — v2 is structurally identical to v1 except where intended
- **B** — zero phantoms survive
- **C** — Bloomberg rows are **bit-identical** to v1 (255,727 rows untouched)
- **D** — every named splice and bad print is actually masked
- **E** — `price_status` reconciles exactly with the change log
- **F** — the 8 splice tickers' *pre-split returns* still compute (only the boundary is nulled)
- **G** — spot-checks: Whirlpool, AHLUCONT, ADANIENT (the quiet case)
- **H** — usable-universe delta vs Cell 1's prediction

In [4]:
# =============================================================================
# STAGE 3B / CELL 3 — v2 DIAGNOSTIC  (gate)
# READ-ONLY. Writes nothing. Mutates nothing.
#
# This is the gate. Nothing downstream consumes v2 until this passes.
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', 60)

BASE = Path('/content/drive/MyDrive/moonshot_compounding')
V1_PANEL = BASE / 'raw' / 'prices' / 'raw_price_panel_consolidated_v1_2026-07-15.parquet'
V2_PANEL = sorted((BASE / 'raw' / 'prices').glob('raw_price_panel_consolidated_v2_*.parquet'))[-1]
CHANGE_LOG = sorted((BASE / 'validation').glob('remediation_change_log_*.csv'))[-1]

MIN_VALID_ROWS = 120

print("=" * 110)
print(f"STAGE 3B / CELL 3 — v2 DIAGNOSTIC")
print(f"  v1  : {V1_PANEL.name}")
print(f"  v2  : {V2_PANEL.name}")
print(f"  log : {CHANGE_LOG.name}")
print("=" * 110)

v1 = pd.read_parquet(V1_PANEL)
v1['Date'] = pd.to_datetime(v1['Date'])
v1 = v1.sort_values(['ticker', 'Date']).reset_index(drop=True)

v2 = pd.read_parquet(V2_PANEL)
v2['Date'] = pd.to_datetime(v2['Date'])
v2 = v2.sort_values(['ticker', 'Date']).reset_index(drop=True)

log = pd.read_csv(CHANGE_LOG)
log['Date'] = pd.to_datetime(log['Date'])

FAILURES = []
def check(name, cond, detail=""):
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f"  — {detail}" if detail else ""))
    if not cond:
        FAILURES.append(name)

# --- A. Structural integrity -------------------------------------------------
print("\n[A] STRUCTURAL INTEGRITY vs v1")
check("row count identical", len(v2) == len(v1), f"{len(v2):,} vs {len(v1):,}")
check("ticker count identical",
      v2['ticker'].nunique() == v1['ticker'].nunique(),
      f"{v2['ticker'].nunique():,} vs {v1['ticker'].nunique():,}")
check("ticker count == 1,069", v2['ticker'].nunique() == 1069)
check("no duplicate (ticker, Date)", not v2.duplicated(['ticker','Date']).any())
check("(Date, ticker) keys identical to v1",
      v1[['Date','ticker']].equals(v2[['Date','ticker']]),
      "the PIT flag joins on these — they MUST be unchanged")
check("price_status column present", 'price_status' in v2.columns)
check("Volume identical to v1 (never masked)",
      v1['Volume'].equals(v2['Volume']),
      "zero shares traded is real information — Q7 source asymmetry")

# --- B. Zero phantoms survive ------------------------------------------------
print("\n[B] PHANTOM ERADICATION")
resid = (
    (v2['Open'] == v2['High']) & (v2['High'] == v2['Low']) &
    (v2['Low'] == v2['Close']) & v2['Open'].notna() & (v2['Volume'] == 0)
)
check("zero phantom rows remain in v2", int(resid.sum()) == 0, f"{int(resid.sum()):,} found")

v1_phantom = (
    (v1['Open'] == v1['High']) & (v1['High'] == v1['Low']) &
    (v1['Low'] == v1['Close']) & v1['Open'].notna() & (v1['Volume'] == 0)
)
print(f"    v1 phantom rows : {int(v1_phantom.sum()):,}")
print(f"    v2 phantom rows : {int(resid.sum()):,}")

# Rows that were phantom in v1 must be NaN in v2 — and only those.
check("every v1 phantom row is NaN in v2",
      v2.loc[v1_phantom.values, ['Open','High','Low','Close']].isna().all().all())

# Rows NOT flagged must be BIT-IDENTICAL. This is the check that catches an
# over-broad mask — the failure mode where the fix damages good data.
untouched = v2['price_status'] == 'observed'
print(f"\n    rows marked 'observed' : {int(untouched.sum()):,}")
for col in ['Open', 'High', 'Low', 'Close']:
    same = v1.loc[untouched.values, col].equals(v2.loc[untouched.values, col])
    check(f"{col} bit-identical to v1 on untouched rows", same)

# --- C. Bloomberg untouched --------------------------------------------------
print("\n[C] BLOOMBERG INTEGRITY  (255,727 rows must be bit-identical)")
b1 = v1[v1['source'] == 'bloomberg'].reset_index(drop=True)
b2 = v2[v2['source'] == 'bloomberg'].reset_index(drop=True)
check("Bloomberg row count unchanged", len(b1) == len(b2), f"{len(b2):,} vs {len(b1):,}")
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    check(f"Bloomberg {col} bit-identical", b1[col].equals(b2[col]))
check("no Bloomberg row was masked",
      (v2.loc[v2['source'] == 'bloomberg', 'price_status'] == 'observed').all(),
      f"{(v2.loc[v2['source']=='bloomberg','price_status'] != 'observed').sum():,} masked")

# --- D. Named events actually masked ----------------------------------------
print("\n[D] NAMED EVENTS — every splice and bad print must be masked")
SPLICE_EVENTS = [
    ('GREENPLY.NS','2016-01-06'), ('MVL.NS','2010-11-16'),
    ('TTML.NS','2013-08-07'), ('IGARASHI.NS','2018-09-27'),
    ('KTKBANK.NS','2020-03-17'), ('KARURVYSYA.NS','2018-08-14'),
    ('JISLJALEQS.NS','2011-11-08'), ('AJRINFRA.NS','2013-01-23'),
]
BAD_PRINTS = [
    ('THYROCARE.NS','2019-10-27'), ('THYROCARE.NS','2020-11-14'),
    ('INDIAMART.NS','2019-10-27'), ('INDIAMART.NS','2020-11-14'),
    ('MVL.NS','2019-10-10'), ('MVL.NS','2019-12-20'),
]
for label, events, expected_status in [
    ("splice", SPLICE_EVENTS, 'split_splice_masked'),
    ("bad print", BAD_PRINTS, 'bad_print_masked'),
]:
    n_ok = 0
    for t, d in events:
        row = v2[(v2['ticker'] == t) & (v2['Date'] == pd.Timestamp(d))]
        if len(row) == 1 and row.iloc[0][['Open','High','Low','Close']].isna().all():
            n_ok += 1
        else:
            print(f"      *** {label} NOT masked: {t} {d}")
    check(f"all {len(events)} {label} rows masked", n_ok == len(events),
          f"{n_ok}/{len(events)}")

print(f"\n    price_status distribution in v2:")
print("    " + v2['price_status'].value_counts().to_string().replace("\n", "\n    "))

# --- E. Change log reconciles ------------------------------------------------
print("\n[E] CHANGE LOG RECONCILIATION")
n_masked_v2 = int((v2['price_status'] != 'observed').sum())
check("change log row count == masked row count in v2",
      len(log) == n_masked_v2, f"log {len(log):,} vs v2 {n_masked_v2:,}")
check("change log status counts match v2",
      log['price_status'].value_counts().sort_index().equals(
          v2.loc[v2['price_status'] != 'observed', 'price_status']
            .value_counts().sort_index()))
print(f"\n    change log by mechanism:")
print("    " + log['price_status'].value_counts().to_string().replace("\n", "\n    "))

# --- F. Splice tickers keep their pre-split history --------------------------
print("\n[F] SPLICE TICKERS — only the boundary is nulled, not the history")
print("    Returns WITHIN the pre-split window are scale-invariant and remain")
print("    valid. Only the single boundary row is lost. Confirming that here.")
for t, d in SPLICE_EVENTS[:4]:
    g = v2[v2['ticker'] == t].sort_values('Date')
    ex = pd.Timestamp(d)
    pre = g[g['Date'] < ex]
    post = g[g['Date'] > ex]
    print(f"      {t:<16} ex {d}  |  pre-split valid Close: "
          f"{pre['Close'].notna().sum():>5,}   post-split: {post['Close'].notna().sum():>5,}")

# --- G. Spot-checks ----------------------------------------------------------
print("\n[G] SPOT-CHECKS")

print("\n  WHIRLPOOL.NS — the original case (4 leading phantoms, +570% exit)")
w = v2[(v2['ticker'] == 'WHIRLPOOL.NS') &
       (v2['Date'] <= pd.Timestamp('2010-01-11'))].sort_values('Date')
print(w[['Date','Open','High','Low','Close','Volume','price_status']]
      .to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print("    Expect: 2010-01-04..07 all NaN OHLC / phantom_masked, Volume 0 retained;")
print("            2010-01-08 untouched with real OHLC and Volume 3,109.")

print("\n  AHLUCONT.NS — the 1,446-row interior run (2011-10-18 -> 2017-09-05)")
a = v2[v2['ticker'] == 'AHLUCONT.NS']
a_run = a[(a['Date'] >= pd.Timestamp('2011-10-18')) & (a['Date'] <= pd.Timestamp('2017-09-05'))]
print(f"    rows in run window        : {len(a_run):,}")
print(f"    of those, phantom_masked  : {int((a_run['price_status']=='phantom_masked').sum()):,}")
print(f"    of those, valid Close     : {int(a_run['Close'].notna().sum()):,}")
print(f"    total valid Close, ticker : {int(a['Close'].notna().sum()):,}  "
      f"(vs v1: {int(v1[v1['ticker']=='AHLUCONT.NS']['Close'].notna().sum()):,})")

print("\n  ADANIENT.NS — the QUIET case (+0.19% artifact, invisible to ±30%)")
ad = v2[(v2['ticker'] == 'ADANIENT.NS') &
        (v2['Date'] >= pd.Timestamp('2026-01-13')) &
        (v2['Date'] <= pd.Timestamp('2026-01-19'))].sort_values('Date')
print(ad[['Date','Open','High','Low','Close','Volume','price_status']]
      .to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print("    A large, liquid index name with a fabricated row. This is why the")
print("    ±30% list was never going to catch this class of error.")

# --- H. Usable universe ------------------------------------------------------
print("\n[H] USABLE UNIVERSE DELTA")
u1 = v1.groupby('ticker')['Close'].apply(lambda s: s.notna().sum())
u2 = v2.groupby('ticker')['Close'].apply(lambda s: s.notna().sum())
usable1 = (u1 >= MIN_VALID_ROWS).sum()
usable2 = (u2 >= MIN_VALID_ROWS).sum()
print(f"    tickers with >= {MIN_VALID_ROWS} valid Close in v1 : {usable1:,} / {len(u1):,}")
print(f"    tickers with >= {MIN_VALID_ROWS} valid Close in v2 : {usable2:,} / {len(u2):,}")
print(f"    tickers lost to the {MIN_VALID_ROWS}-day floor      : {usable1 - usable2:,}")
lost = sorted(set(u1[u1 >= MIN_VALID_ROWS].index) - set(u2[u2 >= MIN_VALID_ROWS].index))
if lost:
    print(f"\n    Tickers crossing the threshold (must match Cell 1's approved list):")
    print(pd.DataFrame({'valid_v1': u1[lost], 'valid_v2': u2[lost]}).to_string())
else:
    print("    None — the usable universe is unchanged.")

# --- Verdict -----------------------------------------------------------------
print("\n" + "=" * 110)
if FAILURES:
    print(f"STAGE 3B DIAGNOSTIC: FAIL — {len(FAILURES)} check(s) failed")
    for f in FAILURES:
        print(f"   - {f}")
    print("DO NOT CONSUME v2 DOWNSTREAM.")
else:
    print("STAGE 3B DIAGNOSTIC: PASS")
    print()
    print("  v2 is safe to consume. Next steps, in the stage4_6 notebook:")
    print("    - The daily PIT flag does NOT need rebuilding. It is built from")
    print("      (Date, ticker) and membership snapshots — it never touches a")
    print("      price. Check [A] confirmed the keys are unchanged.")
    print("    - Rebuild returns from v2 -> returns_panel_v2_<date>.parquet")
    print("    - The +/-30% list should collapse from 193 to ~150. If it does")
    print("      NOT shrink, the mask did not work.")
    print("    - Then resume the original plan at Step 3 (turnover + vol).")
    print("    - KSK.NS vol should now be NaN, not ~0.")
print("=" * 110)

STAGE 3B / CELL 3 — v2 DIAGNOSTIC
  v1  : raw_price_panel_consolidated_v1_2026-07-15.parquet
  v2  : raw_price_panel_consolidated_v2_2026-07-17.parquet
  log : remediation_change_log_2026-07-17.csv

[A] STRUCTURAL INTEGRITY vs v1
  [PASS] row count identical  — 3,332,879 vs 3,332,879
  [PASS] ticker count identical  — 1,069 vs 1,069
  [PASS] ticker count == 1,069
  [PASS] no duplicate (ticker, Date)
  [PASS] (Date, ticker) keys identical to v1  — the PIT flag joins on these — they MUST be unchanged
  [PASS] price_status column present
  [PASS] Volume identical to v1 (never masked)  — zero shares traded is real information — Q7 source asymmetry

[B] PHANTOM ERADICATION
  [PASS] zero phantom rows remain in v2  — 0 found
    v1 phantom rows : 57,876
    v2 phantom rows : 0
  [PASS] every v1 phantom row is NaN in v2

    rows marked 'observed' : 3,274,989
  [PASS] Open bit-identical to v1 on untouched rows
  [PASS] High bit-identical to v1 on untouched rows
  [PASS] Low bit-identical to v1

---

## After this notebook

**Do not rebuild the PIT flag.** `daily_pit_flag_v1_2026-07-16.parquet` is built
from `(Date, ticker)` and the membership snapshots — it never touches a price.
A phantom row still has a real date and a real ticker; only its prices were
fabricated. Every flag on every row is still correct. Check `[A]` confirms the
join keys are unchanged.

**Rebuild from v2, in the `stage4_6` notebook:**

1. `returns_panel_v2_<date>.parquet` — rebuilt from v2
2. Returns diagnostic — the identity check must still hold; the ±30% list
   should collapse from 193 to ~150
3. Resume the original plan at Step 3 (turnover + realized vol)

**Expected changes after remediation:**

| Before | After |
|---|---|
| 47,123 fake zero returns | NaN — vol stops being crushed |
| 21,458 boundary artifacts (10,686 quiet) | NaN — including the invisible ones |
| `KSK.NS` vol ≈ 0 across a 30-month suspension | NaN |
| ±30% list: 193 rows | ~150 rows |

---

## Manifest entries owed

- **v1 superseded by v2** — lineage, reason, change log path
- **Splice tickers: price *levels* not comparable across the split boundary**
  for the 8 named tickers. Returns within each window are fine (scale-invariant);
  only levels are affected. A future hypothesis using price levels must know this.
- **Lesson: thresholds catch loud errors; only structure catches quiet ones.**
  The ±30% list caught 0.9% of the artifacts. Every validation we had was
  threshold-based, and this contamination was sub-threshold by construction.

---

## Still deferred (unchanged by this notebook)

| Item | Where |
|---|---|
| `KENNAMET.NS` 861-day price gap | anomaly review |
| Semi-annual PIT resolution limit (RANBAXY: flag drops 2014-12-31, traded to 2015-04-01) | manifest |
| 5 no-price constituents (Vedanta ×3, Malco, Kwality Wall's) | `validation/pit_constituents_without_prices_*.csv` |
| 2018-07-01 → 2018-12-30 carry-forward window (~22% potential misclassification) | manifest |